In [1]:
import pandas as pd

df = pd.read_csv('dic_data/semantic_mapping_full.csv')

# 1. 检查是否有重复的伪词定义（防止模型偷懒）
duplicates = df['Definition'].duplicated().sum()

# 2. 检查是否有缺失值
missing = df.isnull().sum().sum()

# 3. 统计词性分布（看看是不是全变成 noun 了，或者分布是否均匀）
pos_counts = df['POS'].value_counts()

# 4. 关键词反查（检查定义里是否不小心提到了原词）
conflicts = []
for idx, row in df.iterrows():
    if str(row['Word']).lower() in str(row['Definition']).lower():
        conflicts.append(row['Pseudoword'])

print(f"📊 质检报告：")
print(f"- 重复定义: {duplicates} 个")
print(f"- 缺失数据: {missing} 处")
print(f"- 词性分布: \n{pos_counts}")
print(f"- 语义冲突 (定义包含原词): {conflicts if conflicts else '无'}")

# 抽样 3 个看看语感
print("\n🔍 随机抽样：")
print(df[['Pseudoword', 'Definition']].sample(3).values)

📊 质检报告：
- 重复定义: 5 个
- 缺失数据: 0 处
- 词性分布: 
POS
noun         94
verb          3
adjective     3
Name: count, dtype: int64
- 语义冲突 (定义包含原词): 无

🔍 随机抽样：
[['weemon'
  'A small, nocturnal mammal native to temperate forests, characterized by a dense, water-repellent fur and a diet primarily consisting of insects and berries.']
 ['urlon'
  'A specialized tool used in precision agriculture for measuring soil moisture levels at varying depths, typically consisting of a calibrated probe with digital readout capabilities.']
 ['cibsle'
  'A small, nocturnal mammal native to temperate forests, characterized by its dense, iridescent fur and a distinctive, high-pitched vocalization used for territorial marking.']]


In [2]:
import pandas as pd
import json
from openai import OpenAI

client = OpenAI(api_key="sk-a3e9ba9547704e3191ea42a9290c8f2b", base_url="https://api.deepseek.com")

def fix_semantic_mapping():
    file_path = 'dic_data/semantic_mapping_full.csv'
    df = pd.read_csv(file_path)

    # 1. 找出需要修复的行：重复的定义 OR 词性不是 noun
    mask_dup = df['Definition'].duplicated(keep=False)
    mask_pos = df['POS'].str.lower() != 'noun'
    to_fix = df[mask_dup | mask_pos].copy()

    if to_fix.empty:
        print("✅ 没有发现需要修复的词。")
        return

    print(f"🔧 发现 {len(to_fix)} 个词需要优化（重复或词性不符）...")

    # 2. 定向重刷逻辑
    system_prompt = """You are a psycholinguistics assistant. 
Create a UNIQUE and SCIENTIFIC fictional noun definition. 
Rules:
1. POS MUST be 'noun'.
2. The definition MUST be unique (avoid 'small nocturnal mammals' unless specifically asked). 
3. Fields to consider: Astronomy, Geology, Ancient History, Marine Biology, Architecture, Music Theory.
4. Response format: JSON with keys: "POS", "Definition", "Learning_Sentence", "Test_Sentence"."""

    for idx in to_fix.index:
        pseudo = df.at[idx, 'Pseudoword']
        print(f"正在重塑: {pseudo}...", end='\r')
        
        try:
            response = client.chat.completions.create(
                model="deepseek-chat",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"Create a unique NOUN definition for: {pseudo}"}
                ],
                response_format={'type': 'json_object'}
            )
            res = json.loads(response.choices[0].message.content)
            
            # 更新原 DataFrame
            df.at[idx, 'POS'] = 'noun'
            df.at[idx, 'Definition'] = res.get('Definition')
            df.at[idx, 'Learning_Sentence'] = res.get('Learning_Sentence')
            df.at[idx, 'Test_Sentence'] = res.get('Test_Sentence')
        except Exception as e:
            print(f"\n❌ {pseudo} 修复失败: {e}")

    # 3. 保存覆盖原文件
    df.to_csv(file_path, index=False, encoding='utf-8-sig')
    print(f"\n\n✨ 修复完成！100 个词现在全部为 unique nouns。")

if __name__ == "__main__":
    fix_semantic_mapping()

🔧 发现 15 个词需要优化（重复或词性不符）...
正在重塑: sacar....

✨ 修复完成！100 个词现在全部为 unique nouns。


In [3]:
import pandas as pd

df = pd.read_csv('dic_data/semantic_mapping_full.csv')

# 1. 检查是否有重复的伪词定义（防止模型偷懒）
duplicates = df['Definition'].duplicated().sum()

# 2. 检查是否有缺失值
missing = df.isnull().sum().sum()

# 3. 统计词性分布（看看是不是全变成 noun 了，或者分布是否均匀）
pos_counts = df['POS'].value_counts()

# 4. 关键词反查（检查定义里是否不小心提到了原词）
conflicts = []
for idx, row in df.iterrows():
    if str(row['Word']).lower() in str(row['Definition']).lower():
        conflicts.append(row['Pseudoword'])

print(f"📊 质检报告：")
print(f"- 重复定义: {duplicates} 个")
print(f"- 缺失数据: {missing} 处")
print(f"- 词性分布: \n{pos_counts}")
print(f"- 语义冲突 (定义包含原词): {conflicts if conflicts else '无'}")

# 抽样 3 个看看语感
print("\n🔍 随机抽样：")
print(df[['Pseudoword', 'Definition']].sample(3).values)

📊 质检报告：
- 重复定义: 0 个
- 缺失数据: 0 处
- 词性分布: 
POS
noun    100
Name: count, dtype: int64
- 语义冲突 (定义包含原词): 无

🔍 随机抽样：
[['empay'
  'A specialized tool used in precision agriculture for measuring soil moisture content at multiple depths simultaneously, typically consisting of a multi-pronged sensor array connected to a digital display unit.']
 ['unoun'
  'A small, nocturnal mammal native to temperate forests, characterized by a dense, silvery-gray fur and a diet primarily consisting of lichens and mosses.']
 ['pasale'
  'A small, specialized tool used in precision engineering, typically made of hardened steel, designed for calibrating or adjusting micro-mechanical components by applying controlled rotational force.']]
